In [1]:
!pip install numpy pandas einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 27.6 MB/s  0:00:00 eta 0:00:01


In [41]:
import numpy as np
import pandas as pd
from typing import Callable, List, Union
from einops import einsum
from IPython.display import display

class ProbTable:
    """
    表示任意概率表（联合分布、边缘分布或条件概率分布）

    描述字符串格式示例:
      "A B=1 | C D=1"
      - "|" 左侧为生成/目标变量 (gen-vars/gen-vals)
      - "|" 右侧为条件变量 (cond-vars/cond-vals)
    """
    def __init__(self, description: str, data: Union[np.ndarray, Callable], shape: tuple = None):
        if isinstance(data, Callable):
            self.probs = np.empty(shape)
            def recurse(assignment: list):
                if len(assignment) == len(shape):
                    self.probs[tuple(assignment)] = data(*assignment)
                else:
                    for i in range(shape[len(assignment)]):
                        recurse(assignment + [i])
            recurse([])
        else:
            self.probs = np.array(data)

        self.cond_vars, self.gen_vars = [], []
        self.cond_vals, self.gen_vals = [], []

        items = description.split(" ")
        on_conditioning_side = False
        for item in items:
            if not item: continue
            if item == "|":
                on_conditioning_side = True
            elif on_conditioning_side:
                if "=" in item: self.cond_vals.append(item)
                else: self.cond_vars.append(item)
            else:
                if "=" in item: self.gen_vals.append(item)
                else: self.gen_vars.append(item)

    @property
    def p(self) -> np.ndarray:
        return self.probs

    def to_df(self) -> pd.DataFrame:
        vars_list = self.cond_vars + self.gen_vars
        output = []
        output.append(", ".join([f"{var}={var.lower()}" for var in self.gen_vars] + self.gen_vals))
        if len(self.cond_vars) > 0 or len(self.cond_vals) > 0:
            output.append("|")
            output.append(", ".join([f"{var}={var.lower()}" for var in self.cond_vars] + self.cond_vals))
        prob_str = "P(" + " ".join(output) + ")"

        rows = []
        def recurse(assignment: list):
            if len(assignment) == len(vars_list):
                rows.append(assignment + [self.probs[tuple(assignment)]])
            else:
                for value in range(self.probs.shape[len(assignment)]):
                    recurse(assignment + [value])
        recurse([])

        cols = [var.lower() for var in vars_list] + [prob_str]
        return pd.DataFrame(rows, columns=cols)

    def _repr_html_(self):
        return self.to_df()._repr_html_()

In [4]:
# Joint Distribution
# P(S, R) S: Sunshine R: Rain
P_SR = ProbTable("S R", [[0.20, 0.08],[0.70, 0.02]])
P_SR


,s,r,"P(S=s, R=r)"
0,0,0,0.20
1,0,1,0.08
2,1,0,0.70
3,1,1,0.02


In [6]:
# Marginalization
# P(S) S: Sunshine
P_S = ProbTable("S", einsum(P_SR.p, "s r -> s"))
P_S


,s,P(S=s)
0,0,0.28
1,1,0.72


In [ ]:
# Conditioning
# P(S | R=1)
R1 = np.array([0, 1]) # Mask for R=1
P_SR1 = ProbTable("S R=1", einsum(P_SR.p, R1, "s r, r -> s")) # Compute P(S, R=1) by slicing R=1 from P(S, R)
P_R1 = ProbTable("R=1", einsum(P_SR1.p, "s ->")) #Compute P(R=1) by collapsing S from P(S, R=1)
P_S_given_R1 = ProbTable("S | R=1", P_SR1.p / P_R1.p)
P_S_given_R1

,s,P(S=s | R=1)
0,0,0.8
1,1,0.2


In [ ]:
# Explaining away - Earthquake, Burglary and Alarm
# 1) B: Burglary, E: Earthquake, A: Alarm
# 2) DAG: B -> A <- E
epsilon = 0.05
# 3) Compute the local conditional probabilities for each variable
P_B = ProbTable("B", [1 - epsilon, epsilon]) # P(B)
P_E = ProbTable("E", [1 - epsilon, epsilon]) # P(E)
P_A_given_BE = ProbTable("A | B E", lambda b, e, a: float(a == (b or e)), (2, 2, 2)) # P(A | B E)
# 4) Define joint distribution as the product of all the local conditional probabilities
P_BEA = ProbTable("B E A", einsum(P_B.p, P_E.p, P_A_given_BE.p, "b, e, b e a -> b e a")) # P(B, E, A) = P(A | B E) * P(B) * P(E)
P_BEA

,b,e,a,"P(B=b, E=e, A=a)"
0,0,0,0,0.9025
1,0,0,1,0.0000
2,0,1,0,0.0000
3,0,1,1,0.0475
4,1,0,0,0.0000
5,1,0,1,0.0475
6,1,1,0,0.0000
7,1,1,1,0.0025


In [50]:
# P(B = 1)
P_B = ProbTable("B", einsum(P_BEA.p, "b e a -> b"))
P_B1 = float(P_B.p[1])
print(P_B1)

# P(B = 1 | A = 1) Probability of burgalry when alarm is on
A1 = np.array([0, 1]) # mask for A=1
P_BA1 = ProbTable("B A=1", einsum(P_BEA.p, A1, "b e a, a -> b")) # P(B, A=1)
P_A1 = ProbTable("A=1", einsum(P_BA1.p, "b -> ")) # P(A=1)
P_B_given_A1 = ProbTable("B | A=1", P_BA1.p / P_A1.p)
P_B1_given_A1 = float(P_B_given_A1.p[1])
display(P_B_given_A1)
print(P_B1_given_A1)

# P(B = 1 | A = 1, E = 1) Probability of burgalry when alarm is on and there is earthquake
E1 = np.array([0, 1]) # mask for E=1
P_BA1E1 = ProbTable("B A=1 E=1", einsum(P_BEA.p, A1, E1, "b e a, a, e -> b")) # P(B, A=1, E=1)
P_A1E1 = ProbTable("A=1 E=1", einsum(P_BA1E1.p, "b -> ")) # P(A=1, E=1)
P_B_given_A1E1 = ProbTable("B | A=1 E=1", P_BA1E1.p / P_A1E1.p)
P_B1_given_A1E1 = float(P_B_given_A1E1.p[1])
print(P_B1_given_A1E1)

0.05


,b,P(B=b | A=1)
0,0,0.487179
1,1,0.512821


0.5128205128205129
0.05000000000000001


In [31]:
# Medical dignosis
# C: cold, A: allergy, H: cough, I: itchy eyes
# DAG: C -> H <- A -> I
p_c = ProbTable("C", [0.9, 0.1])
p_a = ProbTable("A", [0.8, 0.2])
p_h_given_a_c = ProbTable("H | A C", lambda a, c, h: 0.9 if h == (a or c) else 0.1, shape=(2, 2, 2))
p_i_given_a = ProbTable("I | A", lambda a, i: 0.9 if i == a else 0.1, shape=(2, 2))
P_ACHI = ProbTable("A C H I", einsum(p_h_given_a_c.p, p_i_given_a.p, p_a.p, p_c.p, "a c h, a i, a, c -> a c h i"))
P_ACHI

,a,c,h,i,"P(A=a, C=c, H=h, I=i)"
0,0,0,0,0,0.5832
1,0,0,0,1,0.0648
2,0,0,1,0,0.0648
3,0,0,1,1,0.0072
4,0,1,0,0,0.0072
5,0,1,0,1,0.0008
6,0,1,1,0,0.0648
7,0,1,1,1,0.0072
8,1,0,0,0,0.0018
9,1,0,0,1,0.0162


In [46]:
# P(C | H=1) Probability of cold if cough
H1 = np.array([0, 1])
P_CH1 = ProbTable("C H=1", einsum(P_ACHI.p, H1, "a c h i, h -> c"))
P_H1 = ProbTable("H=1", einsum(P_CH1.p, "c -> "))
P_C_given_H1 = ProbTable("C | H=1", P_CH1.p / P_H1.p)
display(P_C_given_H1)

# P(C | H=1, I=1) Probably of cold if itchy eyes and cough
I1 = np.array([0, 1])
P_CH1I1 = ProbTable("C H=1 I=1", einsum(P_ACHI.p, H1, I1, "a c h i, h, i -> c"))
P_H1I1 = ProbTable("H=1 I=1", einsum(P_CH1I1.p, "c -> "))
P_C_given_H1I1 = ProbTable("C | H=1 I=1", P_CH1I1.p / P_H1I1.p)
display(P_C_given_H1I1)

print(f"P(Cold=1 | Cough=1):{P_C_given_H1.p[1]:.4f}")
print(f"P(Cold=1 | Cough=1, ItchyEyes=1):{P_C_given_H1I1.p[1]:.4f}")



,c,P(C=c | H=1)
0,0,0.722222
1,1,0.277778


,c,"P(C=c | H=1, I=1)"
0,0,0.867347
1,1,0.132653


P(Cold=1 | Cough=1):0.2778
P(Cold=1 | Cough=1, ItchyEyes=1):0.1327


In [61]:
# Rejection sampling
from collections import defaultdict

def Bernoulli(prob: float) -> int:
    return np.random.choice([0, 1], p=[1 - prob, prob])

def rejection_sampling(program: Callable, evidence: Callable, query: Callable, num_samples: int = 100):
    counts = defaultdict(int)
    for _ in range(num_samples):
        sample = program()
        if evidence(sample):
            counts[query(sample)] += 1
    total_counts = sum(counts.values())
    if total_counts == 0: return {}
    return {q: count / total_counts for q, count in counts.items()}


In [62]:
# Rejection sampling - Earthquake, Burglary and Alarm
# B: Burglary, E: Earthquake, A: Alarm
# DAG: B -> A <- E
# P(B | A = 1)
def alarm_program():
    B = Bernoulli(0.05)
    E = Bernoulli(0.05)
    A = B or E
    return {"A": A, "B": B, "E": E}

evidence = lambda s: s["A"] == 1
query = lambda s: s["B"]
result = rejection_sampling(alarm_program, evidence, query, num_samples=10000)
print(f"Rejection sampling P(B=1 | A=1):{result.get(1, 0):.4f}")

Rejection sampling P(B=1 | A=1):0.5082


In [66]:
# Hidden Markov Model: P (H_3 | E_5 = 2)
def hmm_program():
    num_steps = 5
    H = [0] * num_steps
    E = [0] * num_steps
    for t in range(num_steps):
        H[t] = (H[t - 1] if t > 0 else 0) + Bernoulli(0.5)
        E[t] = H[t] + Bernoulli(0.5)
    return {"H": H, "E": E}

evidence = lambda s: s["E"][4] == 2
query = lambda s: s["H"][2]
result = rejection_sampling(hmm_program, evidence, query, num_samples=20000)
print("P(H_3 | E_5 = 2):")
for val, prob in sorted(result.items()):
    print(f"  H_3 ={val}:{prob:.4f}")

P(H_3 | E_5 = 2):
  H_3 =0:0.1994
  H_3 =1:0.6014
  H_3 =2:0.1992
